In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import math
from collections import Counter
import os

# ==========================================
# 1. 모델 클래스
# ==========================================


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
       
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
             # mask가 0인 부분을 -무한대로 보내서 Softmax 결과가 0이 되게 함
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        return torch.matmul(attn_probs, V)
    

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
       
    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
       
    
    def forward(self, Q, K, V, mask=None): #forward가 되니까 역전파 가능
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        return self.W_o(self.combine_heads(attn_output))


class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    def forward(self, x): # forward -> 역전파 가능
        return self.fc2(self.relu(self.fc1(x)))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask): 
        # Post-NL : 포스트 normalizer -> 현재는 사용하면 문제 있음
        # Post_LN(decoder까지 오차가 전달X,끝까지 전달X) -> Pre-LN(해결됨)
        # 즉 임베딩(디코더)단계로 못가니까 단어이해부터 안됨

        # pre-ln:노멀라이즈로 안전하게 만들어 놓고 더하자(즉,노멀라이즈랑 add랑 순서바꾼거임)
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x))) # Post-LN벙삭
        # Post-LN -> Pre-LN -> peri-ln, mix-ln (검토해보기)
        # 트랜스포머 생성자 -> pre,post선택가능함(결론:pre선택해라)
        # x = x+self.norm2(self.dropout(self.feed_forward(x))) # Pre-ln방식(안정적)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_output, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
                                                    # enc_output:인코더에서 넘어온 key
                                                    # enc_outout:인코더에서 넘어온 valoue
                                                    # src : 영어
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_output, enc_output, src_mask)))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x


# 위에서 나온 class를 조립함, 실제 번역기(위에만들어진 부품을 가져다가 씀)
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout, pad_idx): # init: 부품만 만들고 아직 조립 X
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.pad_idx = pad_idx  # 패딩 인덱스 저장


    def generate_mask(self, src, tgt):
        # src_mask: 패딩 부분은 0, 나머지는 1
        src_mask = (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
       
        # tgt_mask: 패딩 마스크 & Look-ahead 마스크
        tgt_pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool().to(src.device)
        tgt_mask = tgt_pad_mask & nopeak_mask
        return src_mask, tgt_mask


    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))
       
        enc_output = src_embedded # positional embedding이 주입되어있는 상태다
        for enc_layer in self.encoder_layers:
            # 만약 src_embedded를 그대로 enc_output으로 넣으면 초기값만 계속 학습되어짐
            enc_output = enc_layer(enc_output, src_mask)
           
        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)
           
        return self.fc(dec_output)


# ==========================================
# 2. 데이터 처리를 위한 유틸리티
# ==========================================


# 특수 토큰 정의
PAD_TOKEN = "<pad>" # 나머지 공간에 채워진 단어 : 토큰이름이 대문자 -> 상수다!!
SOS_TOKEN = "<sos>" # 문장의 시작 기호
EOS_TOKEN = "<eos>" # 문장의 끝 기호
UNK_TOKEN = "<unk>" # 모르는 단어(단어사전에 없는 단어)

#[1]
class Vocabulary: # 단어사전을 만드는 class , 단어-인덱스 있는 page 만든다고 생각해라
    """단어(토큰)와 정수 인덱스를 매핑하는 클래스"""
    def __init__(self):
        self.token2idx = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
        self.idx2token = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
        self.vocab_size = 4 #단어의 수

    def build_vocab(self, sentences):
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split()) 
            # 문맥파악은 attention 에서 할거고 여기선 많이 나오는 단어를 일렬로 세울려고 함
       
        for word, _ in counter.items():
            if word not in self.token2idx:
                self.token2idx[word] = self.vocab_size
                self.idx2token[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text): # 입력에 사용
        """텍스트 -> 숫자 리스트 (SOS, EOS 포함)"""
        return [self.token2idx[SOS_TOKEN]] + \
               [self.token2idx.get(word, self.token2idx[UNK_TOKEN]) for word in text.split()] + \
               [self.token2idx[EOS_TOKEN]]


    def decode(self, indices): # 맨마지막 출력에 사용
        """숫자 리스트 -> 텍스트 (특수 토큰 제거)"""
        tokens = []
        for idx in indices:
            token = self.idx2token.get(idx, UNK_TOKEN)
            if token == EOS_TOKEN: break
            if token not in [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN]:
                tokens.append(token)
        return " ".join(tokens)


class TranslationDataset(Dataset):
    """PyTorch 데이터셋"""
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
        self.src_data = [torch.tensor(src_vocab.encode(s)) for s in src_sentences] # 영어
        self.tgt_data = [torch.tensor(tgt_vocab.encode(s)) for s in tgt_sentences] # 한글

    # 앞에서 만든 dataset은 오버라이딩 해야함
    def __len__(self):
        return len(self.src_data) # 전체길이


    def __getitem__(self, idx):# 하나씩 꺼내주는 넘
        # src_data:data , tgt_dta:label
        return self.src_data[idx], self.tgt_data[idx] 


def collate_fn(batch):
    """배치 내 문장 길이를 맞추기 위한 패딩 처리"""
    src_batch, tgt_batch = zip(*batch)
    # 0번 인덱스(PAD_TOKEN)로 패딩 채우기
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_padded, tgt_padded


# ==========================================
# 3. 학습 및 추론 실행
# ==========================================


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# 1) 실제 텍스트 데이터 (학습용 예제)
raw_data = [
    ("i love you", "나는 너를 사랑해"),
    ("hello world", "안녕 세상아"),
    ("deep learning is fun", "딥러닝은 재미있어"),
    ("pytorch is powerful", "파이토치는 강력해"),
    ("transformer is all you need", "트랜스포머가 너에게 필요한 전부야"),
    ("good morning", "좋은 아침"),
    ("how are you", "어떻게 지내"),
    ("thank you", "고마워"),
    ("see you later", "나중에 봐"),
    ("i am a student", "나는 학생이야")
]
src_sentences, tgt_sentences = zip(*raw_data) # 영어따로,한글따로 묶어주는 역할
# enumerate, map, fit,zip ---> 엄청많이 쓰임

# 2) 단어장(Vocab) 생성
src_vocab = Vocabulary()
tgt_vocab = Vocabulary()
src_vocab.build_vocab(src_sentences)
tgt_vocab.build_vocab(tgt_sentences)


print(f"Source Vocab Size: {src_vocab.vocab_size}") # 28 영어사전
print(f"Target Vocab Size: {tgt_vocab.vocab_size}") # 25 한국어사전


# 3) 데이터셋 및 데이터로더 준비
dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)


# 4) 모델 초기화
model = Transformer(
    src_vocab_size=src_vocab.vocab_size,
    tgt_vocab_size=tgt_vocab.vocab_size,
    d_model=128,      # 예제라 작게 설정
    num_heads=4, # 수평적인 4개의 attention
    num_layers=2,
    d_ff=256, # 256개의 층을 세우갰다.
    max_seq_length=50, #문장 최대 길이
    dropout=0.1, 
    pad_idx=0         # 패딩 인덱스 전달
).to(device) # gpu 쿠다로 보내겠따.


# 5) 학습 루프 
optimizer = optim.Adam(model.parameters(), lr=0.001)
# 여기서 Adam 반드시 써야 하는 이유:post-ln구조여저
# Adam 이 하는것:warm up 먼저함, 그다음 가속을 주고 간다
# ignore_index=0: 패딩 토큰에 대해서는 Loss를 계산하지 않음 (중요!)
criterion = nn.CrossEntropyLoss(ignore_index=0) # 0은 계산할 필요없음 근데 저게 빠지면 계산을 하기는 한다.


print("\n[Start Training]")
model.train() #학습 모드 전환
for epoch in range(100): # 100 에폭 과적합 학습 (데이터가 적으므로)
                         # 1 에폭마다 5개의 batch를 받는다.
    total_loss = 0 #epoch 마다의 loss 다.
    for src, tgt in dataloader: # Dataset __GETitem()__ return src,tgt 
        src, tgt = src.to(device), tgt.to(device) # cuda로 이동
       
        # Transformer 입력: tgt는 <sos> ... 마지막 단어 (정답에서 마지막 <eos> 제외)
        tgt_input = tgt[:, :-1]
        # 정답 레이블: ... <eos> (입력에서 <sos> 제외)
        tgt_output = tgt[:, 1:]


        optimizer.zero_grad() # 기울기 초기화
        output = model(src, tgt_input)
       
        
        # Loss 계산을 위한 차원 변경
        # output: (batch, seq_len, vocab_size), target: (batch, seq_len)

        loss = criterion(output.reshape(-1, tgt_vocab.vocab_size), tgt_output.reshape(-1))
       
        loss.backward() # 기울기만 구한다
        optimizer.step() # 업데이트 : 학습(역전파)
        total_loss += loss.item()
       
        # 여기까지: 데이터로더 반복 구간

    # 에폭 반복 구간 
    if (epoch+1) % 20 == 0: # 20번마다 epoc 반복해라잉
        print(f"Epoch {epoch+1} | Loss: {total_loss / len(dataloader):.4f}")


# 6) 추론 (Inference) : 예측값뽑기 - Greedy Decoding(단어사전(25)개의 확률높은거 뽑는다)
# beam search:다음다음 확률까지 찾는다.
def translate_sentence(model, sentence, src_vocab, tgt_vocab, device, max_len=50):
    model.eval()
    # 1. 소스 문장 인코딩
    src_tensor = torch.tensor(src_vocab.encode(sentence)).unsqueeze(0).to(device) # (1, seq_len)
    
    # 2. 타겟 문장 시작 (<sos>)
    tgt_indexes = [tgt_vocab.token2idx[SOS_TOKEN]]
   
    for i in range(max_len):
        tgt_tensor = torch.tensor(tgt_indexes).unsqueeze(0).to(device) # (1, current_seq_len)
       
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor) # 모델의 순전파만 태운다.
       
        # 다음 단어 예측 (가장 높은 확률을 가진 단어 인덱스)
        # output shape: (1, seq_len, vocab_size) -> 마지막 시점의 예측값 가져오기
        pred_token = output.argmax(2)[:,-1].item() #argmax: 최댓값을 갖는 인덱스 반환
       
        tgt_indexes.append(pred_token)


In [ ]:
# ==========================================
# 학습된 모델로 번역 테스트 (Inference)
# ==========================================
# 동아 영어사전으로 학습시킬수도 있음
# 현재코드은 있는 입력을 외우기만 한거임
# 학습되지 않은 입력에 대한 답은 안나옴

# 1. 추론(번역) 함수 정의
def translate_sentence(model, sentence, src_vocab, tgt_vocab, device, max_len=50):
    model.eval() # 평가 모드로 전환 (Dropout 자동으로 비활성화)
                 # 추론할때도 eval 이 필요함, 이발안하면 수동으로 dropout 지워야함
                 
    # 입력 문장을 숫자로 변환 (Encoding)
    src_tensor = torch.tensor(src_vocab.encode(sentence)).unsqueeze(0).to(device)
   
    # 디코더의 첫 입력은 <sos> (Start of Sentence)
    tgt_indexes = [tgt_vocab.token2idx["<sos>"]]
   
    for i in range(max_len):
        tgt_tensor = torch.tensor(tgt_indexes).unsqueeze(0).to(device)
       
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor)
       
        # 확률이 가장 높은 다음 단어 선택 (Greedy Decoding)
        pred_token = output.argmax(2)[:,-1].item()
       
        tgt_indexes.append(pred_token)
       
        # <eos> (End of Sentence)를 만나면 번역 종료
        if pred_token == tgt_vocab.token2idx["<eos>"]:
            break
           
    return tgt_vocab.decode(tgt_indexes) # 한국어로 바뀜


# 2. 테스트 문장 번역 실행
print("\n[최종 번역 결과 테스트]")
test_sentences = [
    "i love you",
    "hello world",
    "deep learning is fun",
    "pytorch is powerful"
]


for sent in test_sentences:
    translated = translate_sentence(model, sent, src_vocab, tgt_vocab, device)
    print(f"영어(입력): {sent:<25} -> 한국어(출력): {translated}")
    # sent:<25 -> 왼쪽으로 정렬해라 , >:25 오른쪽정렬, ^ 는 중앙정렬


# 3. 직접 입력해서 테스트해보기
user_input = input("번역할 영어 문장을 입력하세요: ")
print(f"결과: {translate_sentence(model, user_input, src_vocab, tgt_vocab, device)}")




[최종 번역 결과 테스트]
영어(입력): i love you                -> 한국어(출력): 나는 너를 사랑해
영어(입력): hello world               -> 한국어(출력): 안녕 세상아
영어(입력): deep learning is fun      -> 한국어(출력): 딥러닝은 재미있어
영어(입력): pytorch is powerful       -> 한국어(출력): 파이토치는 강력해
결과: 나는 너를 사랑해
